In [2]:
import pandas as pd
import numpy as np
import psycopg2




In [3]:
conn = psycopg2.connect(
    host="localhost",
    port="5432",
    database="analytics_ops_db",
    user="postgres",
    password=""
)

cursor = conn.cursor()

print("Connected to PostgreSQL successfully!")

Connected to PostgreSQL successfully!


In [4]:
cursor.execute("select count(*) as total_suppliers from suppliers")
rows = cursor.fetchone()
rows[0]
# for row in rows:
#     print(row)

50

In [4]:
queries = {
"Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",

"Total Products": "SELECT COUNT(*) AS count FROM products",

"Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",

"Total Sale Value (Last 3 Months)": """
SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) as total_sales_value_in_last_3_month
FROM stock_entries se
JOIN products p
  ON se.product_id = p.product_id
WHERE se.change_type = 'Sale'
  AND se.entry_date >= (
        SELECT MAX(entry_date) - INTERVAL '3 months'
        FROM stock_entries
      );
""",

"Total Restock Value (Last 3 Months)": """
SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) as total_Restock_value_in_last_3_month
FROM stock_entries se
JOIN products p
  ON se.product_id = p.product_id
WHERE se.change_type = 'Restock'
  AND se.entry_date >= (
        SELECT MAX(entry_date) - INTERVAL '3 months'
        FROM stock_entries
      );
""",

"Below Reorder & No Pending Reorders": """
select count(*) from products p where p.stock_quantity<p.reorder_level
and product_id not in (select distinct product_id from reorders where status='Pending' )
"""
}

In [5]:
# conn.rollback()

result = {}

for label, query in queries.items():
    cursor.execute(query)
    row = cursor.fetchone()
    result[label]=row[0]

In [6]:
result

{'Total Suppliers': 50,
 'Total Products': 200,
 'Total Categories Dealing': 5,
 'Total Sale Value (Last 3 Months)': Decimal('1529482.99'),
 'Total Restock Value (Last 3 Months)': Decimal('6691663.10'),
 'Below Reorder & No Pending Reorders': 14}

In [13]:
def get_basic_info(cursor):
    queries = {
    "Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",

    "Total Products": "SELECT COUNT(*) AS count FROM products",

    "Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",

    "Total Sale Value (Last 3 Months)": """
    SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) as total_sales_value_in_last_3_month
    FROM stock_entries se
    JOIN products p
    ON se.product_id = p.product_id
    WHERE se.change_type = 'Sale'
    AND se.entry_date >= (
            SELECT MAX(entry_date) - INTERVAL '3 months'
            FROM stock_entries
        );
    """,

    "Total Restock Value (Last 3 Months)": """
    SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) as total_Restock_value_in_last_3_month
    FROM stock_entries se
    JOIN products p
    ON se.product_id = p.product_id
    WHERE se.change_type = 'Restock'
    AND se.entry_date >= (
            SELECT MAX(entry_date) - INTERVAL '3 months'
            FROM stock_entries
        );
    """,

    "Below Reorder & No Pending Reorders": """
    select count(*) from products p where p.stock_quantity<p.reorder_level
    and product_id not in (select distinct product_id from reorders where status='Pending' )
    """
    }


    result = {}

    for label, query in queries.items():
        cursor.execute(query)
        row = cursor.fetchone()
        result[label]=row[0]

    return result
    

In [14]:
data = get_basic_info(cursor)
data

{'Total Suppliers': 50,
 'Total Products': 200,
 'Total Categories Dealing': 5,
 'Total Sale Value (Last 3 Months)': Decimal('1529482.99'),
 'Total Restock Value (Last 3 Months)': Decimal('6691663.10'),
 'Below Reorder & No Pending Reorders': 14}

In [15]:
queries = {
        "Suppliers Contact Details": "SELECT supplier_name, contact_name, email, phone FROM suppliers",

        "Products with Supplier and Stock": """
            SELECT 
                p.product_name,
                s.supplier_name,
                p.stock_quantity,
                p.reorder_level
            FROM products p
            JOIN suppliers s ON p.supplier_id = s.supplier_id
            ORDER BY p.product_name ASC
        """,

        "Products Needing Reorder": """
            SELECT product_name, stock_quantity, reorder_level
            FROM products
            WHERE stock_quantity <= reorder_level
        """
    }

tables = {}
for label, query in queries.items():
    cursor.execute(query)
    tables[label] = cursor.fetchall()

In [16]:
tables

{'Suppliers Contact Details': [('Anderson-Thompson',
   'Bonnie Davis',
   'zacharysanchez@hotmail.com',
   '829.485.9853x0522'),
  ('Rowland Ltd', 'Beth Stevens', 'joshua60@yahoo.com', '(779)942-0726'),
  ('Baxter-Meadows',
   'Lisa Lewis',
   'andersonchristina@yahoo.com',
   '449-766-7325'),
  ('Wilson, Graham and Williams',
   'David Martinez',
   'abrown@hotmail.com',
   '+1-653-827-5215x266'),
  ('Smith, Kennedy and Moreno',
   'Victoria Gonzalez',
   'mary30@williams-moore.com',
   '891-859-2775x35297'),
  ('Middleton LLC',
   'Megan Miller',
   'kelseywilliams@gmail.com',
   '001-724-731-6199x4596'),
  ('Evans Inc',
   'Danielle Moore',
   'ihoffman@warren.com',
   '+1-951-447-1975x770'),
  ('Lawrence, Garcia and Hernandez',
   'Tamara Johnson',
   'georgeherrera@hotmail.com',
   '+1-635-460-8476x270'),
  ('Young, Browning and Ware',
   'Heather Hill',
   'stephensjason@yahoo.com',
   '(514)361-6411x489'),
  ('Newton, Valencia and Carr',
   'Kimberly Collins',
   'jonathanjohns

In [ ]:
def get_additonal_tables(cursor):
    queries = {
        "Suppliers Contact Details": "SELECT supplier_name, contact_name, email, phone FROM suppliers",

        "Products with Supplier and Stock": """
            SELECT 
                p.product_name,
                s.supplier_name,
                p.stock_quantity,
                p.reorder_level
            FROM products p
            JOIN suppliers s ON p.supplier_id = s.supplier_id
            ORDER BY p.product_name ASC
        """,

        "Products Needing Reorder": """
            SELECT product_name, stock_quantity, reorder_level
            FROM products
            WHERE stock_quantity <= reorder_level
        """
    }

    tables = {}
    for label, query in queries.items():
        cursor.execute(query)
        tables[label] = cursor.fetchall()

    return tables

In [4]:
def add_products(cursor, conn, p_name, p_category, p_price, p_stock, p_reorder, p_supplier):
    procedure_call = "CALL add_new_product(%s, %s, %s, %s, %s)"
    params = (p_name, p_category, p_price, p_stock, p_reorder, p_supplier)
    conn.commit()

In [7]:
def get_categories(cursor):
    cursor.execute("select distinct category from products order by category")
    rows = cursor.fetchall()
    return [row[0] for row in rows]




In [ ]:
data = get_categories(cursor)
data[0]

'C'

In [13]:
def get_suppliers(cursor):
    cursor.execute("select distinct supplier_id, supplier_name from suppliers order by supplier_name")
    return cursor.fetchall()


In [22]:
data = get_suppliers(cursor)

supplier_ids = [s[1] for s in data]
supplier_ids

['Anderson-Thompson',
 'Armstrong-Vance',
 'Barker, White and Carson',
 'Barrett Ltd',
 'Baxter-Meadows',
 'Charles Inc',
 'Clark Group',
 'Douglas Ltd',
 'Elliott-Ayers',
 'Evans Inc',
 'Franklin, Kane and Price',
 'Freeman-Gordon',
 'Gallagher-Miller',
 'Gomez PLC',
 'Hall-Brown',
 'Harris-Cummings',
 'Henderson LLC',
 'Hensley-Branch',
 'Hudson Inc',
 'Johnson-Bass',
 'Kaufman Ltd',
 'Lawrence, Garcia and Hernandez',
 'Lloyd and Sons',
 'Mann-Marshall',
 'Mendoza-Jones',
 'Middleton LLC',
 'Miller-Martinez',
 'Moody-Vang',
 'Morgan-Andrews',
 'Morgan Inc',
 'Moss-Evans',
 'Newton, Valencia and Carr',
 'Ortega-Mahoney',
 'Patrick, Walter and Harrison',
 'Perez, Price and Wallace',
 'Reynolds-Phillips',
 'Rogers-Greene',
 'Rowe PLC',
 'Rowland Ltd',
 'Smith, Kennedy and Moreno',
 'Stewart, Williams and Cox',
 'Taylor-Love',
 'Tran LLC',
 'Tucker-Arnold',
 'Turner-Davis',
 'Vega, Cook and Miller',
 'Williams Ltd',
 'Wilson, Graham and Williams',
 'Wong Group',
 'Young, Browning and War

In [7]:
def get_pending_reorders(cursor):
    cursor.execute("""
    select r.reorder_id , p.product_name
    from reorders as r join products as p 
    on r.product_id= p.product_id
    """)
    return cursor.fetchall()

In [8]:
get_pending_reorders(cursor)

[(1, 'Someone Shirt'),
 (3, 'Space Toy'),
 (4, 'Blue Device'),
 (5, 'Mouth Shirt'),
 (6, 'School Table'),
 (7, 'Four Shirt'),
 (8, 'Fast Shirt'),
 (9, 'Character Table'),
 (10, 'Fact Device'),
 (11, 'Return Table'),
 (12, 'Scene Table'),
 (13, 'Mission Snack'),
 (14, 'Old Shirt'),
 (15, 'Within Toy'),
 (16, 'Lead Toy'),
 (17, 'Guy Device'),
 (18, 'Bag Table'),
 (19, 'However Table'),
 (20, 'White Table'),
 (21, 'Real Snack'),
 (22, 'Investment Toy'),
 (23, 'System Snack'),
 (24, 'Within Toy'),
 (25, 'Appear Toy'),
 (26, 'Four Shirt'),
 (27, 'Thank Shirt'),
 (28, 'Paper Toy'),
 (29, 'Floor Toy'),
 (30, 'Could Device'),
 (31, 'Bed Toy'),
 (32, 'Already Snack'),
 (33, 'Instead Table'),
 (34, 'Wait Toy'),
 (35, 'Still Snack'),
 (36, 'Describe Toy'),
 (37, 'Guy Device'),
 (38, 'Full Table'),
 (39, 'Foreign Table'),
 (40, 'After Table'),
 (41, 'During Toy'),
 (42, 'School Table'),
 (43, 'Final Toy'),
 (44, 'Nor Snack'),
 (45, 'Like Shirt'),
 (46, 'Include Device'),
 (47, 'Determine Table'),
